In [2]:
!pip install transformers==4.44.2 joblib==1.4.2 scikit-learn==1.6.0 numpy==1.26.4 pandas==2.2.3 scipy==1.13.1 seaborn==0.13.2 tqdm==4.66.5 lightgbm==4.5.0 xgboost==2.1.3

In [3]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [4]:
# Load datasets
train_df = pd.read_csv('/kaggle/input/dataset-caco2/Train_Caco2.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/dataset-caco2/Test_Caco2.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [5]:
tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
model = AutoModelForSequenceClassification.from_pretrained('seyonec/ChemBERTa-zinc-base-v1', num_labels=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# dataset class
class SMILESDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=325):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        smiles = self.dataframe.iloc[idx]['SMILES']
        permeability = self.dataframe.iloc[idx]['Permeability']
        inputs = self.tokenizer(smiles, return_tensors='pt', padding="max_length", truncation=True, max_length=self.max_length)
        
        input_ids = inputs['input_ids'].squeeze(0)  # Shape: (sequence_length,)
        attention_mask = inputs['attention_mask'].squeeze(0)  # Shape: (sequence_length,)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(permeability, dtype=torch.float)
        }


# datasets
train_dataset = SMILESDataset(train_df, tokenizer)
test_dataset = SMILESDataset(test_df, tokenizer)
batch_size = 16
# data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at seyonec/ChemBERTa-zinc-base-v1 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 20

In [7]:
# Training loop
from tqdm import tqdm
for epoch in range(num_epochs):
    print(f"Entered Epoch {epoch + 1}")
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f'Training Epoch {epoch + 1}/{num_epochs}', unit='batch'):
        optimizer.zero_grad()

        # Move all batch tensors to device
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"].unsqueeze(1)  # still shape: (batch_size, 1)

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=labels
        )
        loss = outputs.loss
        train_loss += loss.item()

        # Backprop and optimizer step
        loss.backward()
        optimizer.step()

    avg_train_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}')

Entered Epoch 1


Training Epoch 1/20: 100%|██████████| 63/63 [00:28<00:00,  2.25batch/s]


Epoch 1/20 - Train Loss: 2.1625
Entered Epoch 2


Training Epoch 2/20: 100%|██████████| 63/63 [00:28<00:00,  2.18batch/s]


Epoch 2/20 - Train Loss: 0.5070
Entered Epoch 3


...

Epoch 18/20 - Train Loss: 0.0957
Entered Epoch 19


Training Epoch 19/20: 100%|██████████| 63/63 [00:34<00:00,  1.80batch/s]


Epoch 19/20 - Train Loss: 0.0957
Entered Epoch 20


Training Epoch 20/20: 100%|██████████| 63/63 [00:34<00:00,  1.80batch/s]

Epoch 20/20 - Train Loss: 0.0802


In [8]:
model_name = 'ChemBERTa_model_1_caco2'
model_save_path = f'/kaggle/working/{model_name}'
os.makedirs(model_save_path, exist_ok=True)

tokenizer.save_pretrained(model_save_path)
model.save_pretrained(model_save_path)

print(f'Model and tokenizer saved to {model_save_path}')

Model and tokenizer saved to /kaggle/working/ChemBERTa_model_1_caco2


In [9]:
from scipy.stats import pearsonr, spearmanr

model.eval()
test_loss = 0
test_true_labels = []
predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing', unit='batch'):
      
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].unsqueeze(1).to(device).float()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        test_loss += loss.item()

        test_true_labels.extend(labels.cpu().numpy())
        preds = outputs.logits.squeeze().cpu().numpy()  
        predictions.extend(preds)

# Final test loss
avg_test_loss = test_loss / len(test_loader)
print(f'Test Loss: {avg_test_loss:.4f}')

test_true_labels = np.array(test_true_labels).flatten()
predictions = np.array(predictions)
print(test_true_labels.shape)
print(predictions.shape)

# Performance metrics
mse = mean_squared_error(test_true_labels, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_true_labels, predictions)
r2 = r2_score(test_true_labels, predictions)
PCC,_ = pearsonr(test_true_labels, predictions)
SCC,_ = spearmanr(test_true_labels, predictions)
# Print performance metrics
print(f'Mean Squared Error: {mse:.4f}')
print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R^2 Score: {r2:.4f}')
print(f'Pearson Correlation Coefficient: {PCC:.4f}')
print(f'Spearman Correlation Coefficient: {SCC:.4f}')

# Print hyperparameters
print("Hyperparameters:")
print(f"Learning Rate: {5e-5}")
print(f"Batch Size: 16")
print(f"Epochs: {num_epochs}")

Testing: 100%|██████████| 16/16 [00:03<00:00,  5.07batch/s]


Test Loss: 0.3220
(252,)
(252,)
Mean Squared Error: 0.3222
Root Mean Squared Error: 0.5677
Mean Absolute Error: 0.4449
R^2 Score: 0.4829
Pearson Correlation Coefficient: 0.7185
Spearman Correlation Coefficient: 0.6814
Hyperparameters:
Learning Rate: 5e-05
Batch Size: 16
Epochs: 20


In [10]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_name = 'ChemBERTa_model_1_caco2'
model_save_path = f'/kaggle/working/{model_name}'

if not os.path.exists(model_save_path):
    raise FileNotFoundError(f"The model directory {model_save_path} does not exist.")

tokenizer = AutoTokenizer.from_pretrained(model_save_path)
model = AutoModel.from_pretrained(model_save_path).to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at /kaggle/working/ChemBERTa_model_1_caco2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# Load your datasets
train_df = pd.read_csv('/kaggle/input/dataset-caco2/Train_Caco2.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/dataset-caco2/Test_Caco2.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [12]:
train_encodings = tokenizer(list(train_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")
test_encodings = tokenizer(list(test_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")

In [13]:
from tqdm import tqdm 
batch_size = 16 

def generate_embeddings(encodings, batch_size):
    embeddings = []
    model.eval() 
    with torch.no_grad():
        for i in tqdm(range(0, len(encodings['input_ids']), batch_size), desc="Processing batches"):
            batch = {key: val[i:i + batch_size].to(device) for key, val in encodings.items()}  
            outputs = model(**batch)
            embeddings.append(outputs.last_hidden_state)
    return torch.cat(embeddings, dim=0)

In [14]:
train_embeddings = generate_embeddings(train_encodings, batch_size)
print(train_embeddings.shape)
train_embeddings = torch.mean(train_embeddings, dim=1)
print(train_embeddings.shape)

Processing batches: 100%|██████████| 63/63 [00:08<00:00,  7.49it/s]


torch.Size([1007, 213, 768])
torch.Size([1007, 768])


In [15]:
column_names = [f'x_fine_emb_ChemBerta{i}' for i in range(train_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=train_embeddings.cpu().numpy(), columns=column_names)
train_data = pd.concat([train_df, embeddings_df], axis=1)

In [16]:
test_embeddings = generate_embeddings(test_encodings, batch_size)
print(test_embeddings.shape)
test_embeddings = torch.mean(test_embeddings, dim=1)
print(test_embeddings.shape)

Processing batches: 100%|██████████| 16/16 [00:02<00:00,  7.86it/s]

torch.Size([252, 216, 768])
torch.Size([252, 768])


In [17]:
column_names = [f'x_fine_emb_ChemBerta{i}' for i in range(test_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=test_embeddings.cpu().numpy(), columns=column_names)
test_data = pd.concat([test_df, embeddings_df], axis=1)

In [18]:
train_data.to_csv("/kaggle/working/Train_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_caco2.csv",index=False)
test_data.to_csv("/kaggle/working/Test_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_caco2.csv",index=False)

In [19]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [20]:
train_data = pd.read_csv("/kaggle/working/Train_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_caco2.csv")
test_data = pd.read_csv("/kaggle/working/Test_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_caco2.csv")

In [21]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.4)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.4)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [22]:
X_train = train_data.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_data['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

X_test = test_data.drop(['ID','SMILES','Permeability'],axis=1)
y_test = test_data['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 768)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 768)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008527 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 768
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.0637,0.1858,0.2524,0.9006,0.9490,0.9500,0.2929,0.4066,0.5412,0.5299,0.7390,0.6944
DecisionTreeRegressor,0.1440,0.2783,0.3795,0.7753,0.8879,0.8957,0.3091,0.4191,0.5560,0.5039,0.7253,0.6810
RandomForestRegressor,0.0642,0.1889,0.2534,0.8998,0.9486,0.9495,0.2950,0.4098,0.5432,0.5265,0.7350,0.6886
GradientBoostingRegressor,0.0650,0.1859,0.2549,0.8987,0.9480,0.9483,0.2963,0.4090,0.5444,0.5244,0.7361,0.6928
AdaBoostRegressor,0.0760,0.2067,0.2758,0.8813,0.9390,0.9366,0.2966,0.4182,0.5446,0.5240,0.7315,0.6885
XGBRegressor,0.0761,0.2063,0.2758,0.8813,0.9390,0.9400,0.2884,0.4065,0.5371,0.5371,0.7427,0.6975
ExtraTreesRegressor,0.0666,0.1881,0.2581,0.8961,0.9466,0.9475,0.2888,0.4041,0.5374,0.5365,0.7413,0.6953
LinearRegression,3.3631,1.4305,1.8339,-4.2476,0.3163,0.3301,1.2803,0.8630,1.1315,-1.0547,0.2639,0.2784
KNeighborsRegressor,0.0896,0.2268,0.2994,0.8601,0.9278,0.9219,0.2860,0.3933,0.5348,0.5411,0.7477,0.7156
SVR,0.0588,0.1767,0.2424,0.9083,0.9532,0.9580,0.2800,0.3980,0.5292,0.5506,0.7494,0.7121


In [23]:
result_df

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.0637,0.1858,0.2524,0.9006,0.9490,0.9500,0.2929,0.4066,0.5412,0.5299,0.7390,0.6944
DecisionTreeRegressor,0.1440,0.2783,0.3795,0.7753,0.8879,0.8957,0.3091,0.4191,0.5560,0.5039,0.7253,0.6810
RandomForestRegressor,0.0642,0.1889,0.2534,0.8998,0.9486,0.9495,0.2950,0.4098,0.5432,0.5265,0.7350,0.6886
GradientBoostingRegressor,0.0650,0.1859,0.2549,0.8987,0.9480,0.9483,0.2963,0.4090,0.5444,0.5244,0.7361,0.6928
AdaBoostRegressor,0.0760,0.2067,0.2758,0.8813,0.9390,0.9366,0.2966,0.4182,0.5446,0.5240,0.7315,0.6885
XGBRegressor,0.0761,0.2063,0.2758,0.8813,0.9390,0.9400,0.2884,0.4065,0.5371,0.5371,0.7427,0.6975
ExtraTreesRegressor,0.0666,0.1881,0.2581,0.8961,0.9466,0.9475,0.2888,0.4041,0.5374,0.5365,0.7413,0.6953
LinearRegression,3.3631,1.4305,1.8339,-4.2476,0.3163,0.3301,1.2803,0.8630,1.1315,-1.0547,0.2639,0.2784
KNeighborsRegressor,0.0896,0.2268,0.2994,0.8601,0.9278,0.9219,0.2860,0.3933,0.5348,0.5411,0.7477,0.7156
SVR,0.0588,0.1767,0.2424,0.9083,0.9532,0.9580,0.2800,0.3980,0.5292,0.5506,0.7494,0.7121


In [24]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-7.102031496060871, -7.382649246147564, -7.01...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.001181849565262, -5.607117109101056, -5.9...","[-7.020502919474867, -5.613484869623124, -5.94...","[0.02286132128264935, 0.021368493858175356, 0...."
1,DecisionTreeRegressor,"[-7.54, -7.21, -7.1, -7.1, -6.11, -7.24, -6.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.68, -5.6, -5.89, -5.85, -5.89, -7.0, -5.7...","[-6.984, -5.588, -6.0311863912, -5.93778707859...","[0.16977632343763385, 0.12432216214336057, 0.2..."
2,RandomForestRegressor,"[-7.16637121025, -7.322090266029999, -7.125991...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.94808607315, -5.6168250096299985, -5.9441...","[-7.012854526745999, -5.605688828716, -5.94794...","[0.043861911094175754, 0.02432840645807968, 0...."
3,GradientBoostingRegressor,"[-7.1293109598530675, -7.322752108147486, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.976706693172081, -5.593763148221166, -5.9...","[-7.057350208815859, -5.6066802468171755, -5.9...","[0.06552931157905817, 0.017084629849411625, 0...."
4,AdaBoostRegressor,"[-7.12830749757408, -7.504294874673468, -7.262...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.05028345032353, -5.640050505050509, -5.98...","[-7.0682341560248645, -5.684748378183999, -5.9...","[0.06544603769376833, 0.0440371550065892, 0.02..."
5,XGBRegressor,"[-7.190769, -7.3997774, -7.151253, -7.086952, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.061006, -5.769258, -5.9048967, -5.838167,...","[-7.0511923, -5.6723, -5.9552965, -5.8798523, ...","[0.027780853, 0.05169743, 0.092295766, 0.07201..."
6,ExtraTreesRegressor,"[-7.117878740809998, -7.340623646079997, -7.04...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.969867874870004, -5.681533594929999, -5.9...","[-7.00085885168, -5.621964639522, -5.968446610...","[0.034503070042320956, 0.03170438211952919, 0...."
7,LinearRegression,"[-9.352186643307611, -8.552792945403887, -10.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-8.111988656652208, -4.440580566683017, -8.1...","[-5.858035335563095, -4.768808497064587, -5.55...","[1.537594413881126, 1.4355515051667407, 1.7166..."
8,KNeighborsRegressor,"[-7.126666666666668, -7.343333333333334, -7.03...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.046666666666667, -5.503333333333333, -5.9...","[-7.1, -5.506666666666667, -5.917999999999999,...","[0.05970296847263444, 0.004216370213558218, 0...."
9,SVR,"[-7.088836903568443, -7.247008016669973, -7.01...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.880783561611995, -5.564457429530786, -5.9...","[-6.882081983240072, -5.558109398843085, -5.97...","[0.028223482022848485, 0.027274114856065946, 0..."


In [25]:
result_df.to_csv('/kaggle/working/Results_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_caco2.csv')
prediction_df.to_csv('/kaggle/working/Prediction_data_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_caco2.csv')